### 검증용 데이터셋을 통해 전체 인식 과정 파이프라인의 정확도를 확인하는 코드

In [ ]:
import os
import cv2
import numpy as np
from ultralytics import YOLO
import sys
from datetime import date, datetime
import calendar  # ✅ 추가

import torch
from torchvision import transforms

# 🔁 (NEW) TrOCR
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image


########################################
# 0. 디바이스
########################################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
to_tensor = transforms.ToTensor()


########################################
# 1. 파인튜닝된 TrOCR 로드
########################################
TROCR_DIR = "/Users/hyeongmin_k/세븐일레븐_데이터_코드_제출용/ocr_moel/best_checkpoint" # 파인튜닝된 TrOCR 모델 경로

processor = TrOCRProcessor.from_pretrained(TROCR_DIR)
trocr_model = VisionEncoderDecoderModel.from_pretrained(TROCR_DIR).to(device).eval()


########################################
# 2. TrOCR로 OCR 하는 함수
########################################
def infer_ocr_from_np(np_img):
    if len(np_img.shape) == 2:  # gray
        img_rgb = Image.fromarray(np_img).convert("RGB")
    else:  # BGR
        img_rgb = Image.fromarray(cv2.cvtColor(np_img, cv2.COLOR_BGR2RGB))

    with torch.no_grad():
        pixel_values = processor(images=img_rgb, return_tensors="pt").pixel_values.to(device)
        generated_ids = trocr_model.generate(pixel_values, max_length=16)
        pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return pred_text.strip()


########################################
# 3. 날짜 후처리 유틸
########################################
MONTH_MAP = {
    "JAN": 1, "JANUARY": 1,
    "FEB": 2, "FEBRUARY": 2,
    "MAR": 3, "MARCH": 3,
    "APR": 4, "APRIL": 4,
    "MAY": 5,
    "JUN": 6, "JUNE": 6,
    "JUL": 7, "JULY": 7,
    "AUG": 8, "AUGUST": 8,
    "SEP": 9, "SEPT": 9, "SEPTEMBER": 9,
    "OCT": 10, "OCTOBER": 10,
    "NOV": 11, "NOVEMBER": 11,
    "DEC": 12, "DECEMBER": 12,
}

TODAY = date(2025, 11, 3)


def clean_token_text(s: str) -> str:
    if not s:
        return ""
    s = s.strip().replace("\n", "").replace("\r", "")
    s = s.replace(".", "").replace(",", "")
    return s


def normalize_month_from_token(tok) -> int | None:
    cls_ = tok["cls"]
    txt = clean_token_text(tok["text"]).upper()
    if cls_ == "ENG_MON":
        if txt in MONTH_MAP:
            return MONTH_MAP[txt]
        if len(txt) >= 3 and txt[:3] in MONTH_MAP:
            return MONTH_MAP[txt[:3]]
        return None
    if cls_ == "NUM2" and txt.isdigit():
        v = int(txt)
        if 1 <= v <= 12:
            return v
    return None


def expand_2digit_year(yy: int, today: date) -> int:
    base = (today.year // 100) * 100
    y = base + yy
    if y < today.year - 1:
        y += 100
    return y


def make_date_safe(y, m, d):
    try:
        return date(y, m, d)
    except ValueError:
        return None


def last_day_of_month(y: int, m: int) -> int:
    return calendar.monthrange(y, m)[1]


########################################
# 3-1. 2토큰 파서 (여기에 MM-DD → 가까운 미래 연도 설계를 넣음)
########################################
def parse_two_tokens(tokens: list[dict], today: date):
    if len(tokens) != 2:
        return None

    toks = sorted(tokens, key=lambda x: x["cx"])
    t1, t2 = toks

    c1, v1 = t1["cls"], clean_token_text(t1["text"]).upper()
    c2, v2 = t2["cls"], clean_token_text(t2["text"]).upper()

    # 1) Y4 NUM2 → YYYY-MM-(마지막날)
    if c1 == "Y4" and c2 == "NUM2" and v1.isdigit() and v2.isdigit():
        y = int(v1)
        m = int(v2)
        if 1 <= m <= 12:
            dt = make_date_safe(y, m, 1)
            if dt and dt >= today:
                return dt, True
        return None

    # 2) Y4 ENG_MON → YYYY-MON-(마지막날)
    if c1 == "Y4" and c2 == "ENG_MON" and v1.isdigit():
        y = int(v1)
        m = normalize_month_from_token(t2)
        if m is not None:
            dt = make_date_safe(y, m, 1)
            if dt and dt >= today:
                return dt, True
        return None

    # ✅ (NEW) 2-1) NUM2 Y4 → MM-YYYY  (월-연도) → YYYY-MM-(마지막날)
    if c1 == "NUM2" and c2 == "Y4" and v1.isdigit() and v2.isdigit():
        m = int(v1)
        y = int(v2)
        if 1 <= m <= 12:
            dt = make_date_safe(y, m, 1)
            if dt and dt >= today:
                return dt, True
        return None

    # ✅ (NEW) 2-2) ENG_MON Y4 → MON-YYYY → YYYY-MM-(마지막날)
    if c1 == "ENG_MON" and c2 == "Y4" and v2.isdigit():
        m = normalize_month_from_token(t1)
        y = int(v2)
        if m is not None:
            dt = make_date_safe(y, m, 1)
            if dt and dt >= today:
                return dt, True
        return None

    # 3) NUM2 NUM2 → MM-DD → 올해가 과거면 내년으로 밀어
    if c1 == "NUM2" and c2 == "NUM2" and v1.isdigit() and v2.isdigit():
        m = int(v1)
        d = int(v2)
        if not (1 <= m <= 12):
            return None
        dt_this_year = make_date_safe(today.year, m, d)
        if dt_this_year and dt_this_year >= today:
            return dt_this_year, False
        # 올해 걸리면 안 되니까 내년으로
        dt_next_year = make_date_safe(today.year + 1, m, d)
        if dt_next_year:
            return dt_next_year, False
        return None

    # 4) ENG_MON NUM2 → MON-DD → 올해가 과거면 내년으로 밀어
    if c1 == "ENG_MON" and c2 == "NUM2" and v2.isdigit():
        m = normalize_month_from_token(t1)
        if m is None:
            return None
        d = int(v2)
        dt_this_year = make_date_safe(today.year, m, d)
        if dt_this_year and dt_this_year >= today:
            return dt_this_year, False
        dt_next_year = make_date_safe(today.year + 1, m, d)
        if dt_next_year:
            return dt_next_year, False
        return None

    return None


########################################
# 3-2. 3토큰 파서 (기존 네 로직)
########################################
def parse_three_tokens(tokens: list[dict], today: date):
    if len(tokens) != 3:
        return None

    # 1) x좌표 기준 정렬
    toks = sorted(tokens, key=lambda x: x["cx"])
    t1, t2, t3 = toks

    c1, v1 = t1["cls"], clean_token_text(t1["text"])
    c2, v2 = t2["cls"], clean_token_text(t2["text"])
    c3, v3 = t3["cls"], clean_token_text(t3["text"])

    # (NEW) 4자리여야 하는데 실제는 2자리 → NUM2로 내려
    if c1 == "Y4" and v1.isdigit() and len(v1) <= 2:
        c1 = "NUM2"
    if c2 == "Y4" and v2.isdigit() and len(v2) <= 2:
        c2 = "NUM2"
    if c3 == "Y4" and v3.isdigit() and len(v3) <= 2:
        c3 = "NUM2"

    # ─── 1) 딱 떨어지는 5패턴 ───
    if c1 == "Y4" and c2 == "NUM2" and c3 == "NUM2" and v1.isdigit() and v2.isdigit() and v3.isdigit():
        y = int(v1); m = int(v2); d = int(v3)
        dt = make_date_safe(y, m, d)
        if dt:
            return dt, True

    if c1 == "NUM2" and c2 == "NUM2" and c3 == "Y4" and v1.isdigit() and v2.isdigit() and v3.isdigit():
        d = int(v1); m = int(v2); y = int(v3)
        dt = make_date_safe(y, m, d)
        if dt:
            return dt, True

    if c1 == "Y4" and c2 == "ENG_MON" and c3 == "NUM2" and v1.isdigit() and v3.isdigit():
        y = int(v1); m = normalize_month_from_token(t2); d = int(v3)
        dt = make_date_safe(y, m, d)
        if dt:
            return dt, True

    if c1 == "NUM2" and c2 == "ENG_MON" and c3 == "Y4" and v1.isdigit() and v3.isdigit():
        d = int(v1); m = normalize_month_from_token(t2); y = int(v3)
        dt = make_date_safe(y, m, d)
        if dt:
            return dt, True

    if c1 == "ENG_MON" and c2 == "NUM2" and c3 == "Y4" and v2.isdigit() and v3.isdigit():
        m = normalize_month_from_token(t1); d = int(v2); y = int(v3)
        dt = make_date_safe(y, m, d)
        if dt:
            return dt, True

    # ─── 2) 가운데가 월로 보이는 케이스 ───
    mid_month_num = normalize_month_from_token(t2)
    if mid_month_num is not None and v1.isdigit() and v3.isdigit():
        cands = []

        # (가) 왼쪽=연도2자리, 오른쪽=일
        yy_left = int(v1); dd_right = int(v3)
        y_full_left = expand_2digit_year(yy_left, today)
        dt_left = make_date_safe(y_full_left, mid_month_num, dd_right)
        if dt_left and dt_left > today:
            cands.append((dt_left, False))

        # (나) 왼쪽=일, 오른쪽=연도2자리
        dd_left = int(v1); yy_right = int(v3)
        y_full_right = expand_2digit_year(yy_right, today)
        dt_right = make_date_safe(y_full_right, mid_month_num, dd_left)
        if dt_right and dt_right > today:
            cands.append((dt_right, False))

        if not cands:
            return None

        cands.sort(key=lambda x: (x[0] - today).days)
        return cands[0]

    # ─── 3) 앞이 월인 경우 ───
    front_month = normalize_month_from_token(t1)
    if front_month is not None:
        if not v2.isdigit():
            return None
        day = int(v2)

        if c3 == "NUM2" and v3.isdigit():
            yy = int(v3)
            y_full = expand_2digit_year(yy, today)
            dt = make_date_safe(y_full, front_month, day)
            if dt:
                return dt, False

        if c3 == "Y4" and v3.isdigit():
            y_full = int(v3)
            dt = make_date_safe(y_full, front_month, day)
            if dt:
                return dt, True

    return None


########################################
# 3-3. 단일 date entry 해석
########################################
def interpret_single_date_entry_v2(entry, today: date):
    tokens = entry["tokens"]

    if len(tokens) == 0:
        return None
    elif len(tokens) == 1:
        return None
    elif len(tokens) == 2:
        out = parse_two_tokens(tokens, today)
        if not out:
            return None
        dt, has4 = out
        return {
            "date_idx": entry["date_idx"],
            "bbox": entry["bbox"],
            "date": dt,
            "has4": has4,
        }
    elif len(tokens) == 3:
        out = parse_three_tokens(tokens, today)
        if not out:
            return None
        dt, has4 = out
        return {
            "date_idx": entry["date_idx"],
            "bbox": entry["bbox"],
            "date": dt,
            "has4": has4,
        }
    else:
        top3 = sorted(tokens, key=lambda x: x["cx"])[:3]
        out = parse_three_tokens(top3, today)
        if not out:
            return None
        dt, has4 = out
        return {
            "date_idx": entry["date_idx"],
            "bbox": entry["bbox"],
            "date": dt,
            "has4": has4,
        }


########################################
# 3-4. 여러 date 중 최종 선택
########################################
def postprocess_all_dates(all_date_results, today: date):
    interpreted = []
    for ent in all_date_results:
        one = interpret_single_date_entry_v2(ent, today)
        if one is None:
            print(f"[DROP] date#{ent['date_idx']}: 규칙에 안 맞아서 버림")
        else:
            interpreted.append(one)

    if not interpreted:
        print("[RETAKE] 유통기한 파싱 실패. 다시 촬영.")
        return None, []

    # 1개면 그대로
    if len(interpreted) == 1:
        final_dt = interpreted[0]["date"]
        print(f"[FINAL] {final_dt.isoformat()}")
        return final_dt, []

    # 2개일 때
    if len(interpreted) == 2:
        a, b = interpreted[0], interpreted[1]

        # ── (1) 둘 다 4자리
        if a["has4"] and b["has4"]:
            chosen = a if a["date"] >= b["date"] else b
            other = b if chosen is a else a
            print("[2CAND/4digit]")
            print(f"  - cand1(idx={a['date_idx']}): {a['date'].isoformat()}")
            print(f"  - cand2(idx={b['date_idx']}): {b['date'].isoformat()}")
            print(f"[FINAL] 두 개 모두 4자리 → 더 나중 날짜: {chosen['date'].isoformat()}")
            return chosen["date"], [other["date"]]

        # ── (2) 둘 다 2자리 → y기준 선택 ❌ → 날짜 기준으로만 선택
        if (not a["has4"]) and (not b["has4"]):
            da = a["date"]
            db = b["date"]

            print("[2CAND/2digit]")
            print(f"  - cand1(idx={a['date_idx']}): {da.isoformat()}")
            print(f"  - cand2(idx={b['date_idx']}): {db.isoformat()}")

            # 2-1) 미래 후보만 골라보기
            futures = []
            if da >= today:
                futures.append(("a", da))
            if db >= today:
                futures.append(("b", db))

            if futures:
                # 미래 중 오늘과 가장 가까운 것
                futures.sort(key=lambda x: (x[1] - today).days)
                picked_key, picked_date = futures[0]
            else:
                # 둘 다 과거 → 덜 지난 것
                past_a = (today - da).days
                past_b = (today - db).days
                if past_a <= past_b:
                    picked_key, picked_date = "a", da
                else:
                    picked_key, picked_date = "b", db

            if picked_key == "a":
                chosen = a
                other = b
            else:
                chosen = b
                other = a

            print(f"[FINAL] 두 개 모두 2자리 → 날짜 기준으로 선택: {chosen['date'].isoformat()}")
            return chosen["date"], [other["date"]]

        # ── (3) 하나만 4자리 → 4자리 우선
        chosen = a if a["has4"] else b
        other = b if chosen is a else a
        print("[2CAND/mixed]")
        print(f"  - cand1(idx={a['date_idx']}): {a['date'].isoformat()} (has4={a['has4']})")
        print(f"  - cand2(idx={b['date_idx']}): {b['date'].isoformat()} (has4={b['has4']})")
        print(f"[FINAL] 4자리/2자리 혼재 → 4자리 우선: {chosen['date'].isoformat()}")
        return chosen["date"], [other["date"]]

    # 3개 이상은 기존 로직
    with_four = [it for it in interpreted if it["has4"]]
    if with_four:
        chosen = max(with_four, key=lambda x: x["date"])
        others = [it["date"] for it in interpreted if it is not chosen]
        print(f"[FINAL] 4자리 연도 후보 중 가장 나중 날짜 선택: {chosen['date'].isoformat()}")
        return chosen["date"], others

    future_all = [it for it in interpreted if it["date"] >= today]
    if future_all:
        chosen = min(future_all, key=lambda x: (x["date"] - today).days)
        others = [it["date"] for it in interpreted if it is not chosen]
        print(f"[FINAL] 2자리 후보 중 가장 가까운 미래 선택: {chosen['date'].isoformat()}")
        return chosen["date"], others

    chosen = min(interpreted, key=lambda x: (today - x["date"]).days)
    others = [it["date"] for it in interpreted if it is not chosen]
    print(f"[FINAL] 2자리 후보 전부 과거 → 가장 가까운 과거 선택: {chosen['date'].isoformat()}")
    return chosen["date"], others



########################################
# 4. YOLO 관련 유틸 (이 아래는 거의 그대로)
########################################
def enlarge_box(x1, y1, x2, y2, scale, img_w, img_h):
    bw = x2 - x1
    bh = y2 - y1
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    new_w = bw * scale
    new_h = bh * scale
    new_x1 = int(round(cx - new_w / 2.0))
    new_y1 = int(round(cy - new_h / 2.0))
    new_x2 = int(round(cx + new_w / 2.0))
    new_y2 = int(round(cy + new_h / 2.0))
    new_x1 = max(0, new_x1)
    new_y1 = max(0, new_y1)
    new_x2 = min(img_w, new_x2)
    new_y2 = min(img_h, new_y2)
    return new_x1, new_y1, new_x2, new_y2


def full_model_class_to_name(cls_id: int) -> str:
    mapping = {0: "date", 1: "due", 2: "code", 3: "full"}
    return mapping.get(int(cls_id), f"cls{int(cls_id)}")


def after_class_to_name(cls_id: int) -> str:
    mapping = {0: "date", 1: "due", 2: "code"}
    return mapping.get(int(cls_id), f"A{int(cls_id)}")


def after_class_to_color(cls_id: int):
    palette = {
        0: (0, 255, 255),
        1: (255, 0, 255),
        2: (255, 255, 0),
    }
    return palette.get(int(cls_id), (200, 200, 200))


def token_class_to_name(cls_id: int) -> str:
    if cls_id == 0:
        return "Y4"
    if cls_id == 1:
        return "ENG_MON"
    if cls_id == 2:
        return "NUM2"
    return f"T{cls_id}"


def token_class_to_color(cls_id: int):
    if cls_id == 0:  # Y4
        return (0, 0, 255)
    if cls_id == 1:  # ENG_MON
        return (0, 255, 0)
    if cls_id == 2:  # NUM2
        return (255, 0, 0)
    return (0, 255, 255)


# YOLO 모델 로드 (full, date, dmy 각각 모델 경로)
model_full = YOLO("/Users/hyeongmin_k/세븐일레븐_데이터_코드_제출용/yolo_model/full_detection.pt")
model_after = YOLO("/Users/hyeongmin_k/세븐일레븐_데이터_코드_제출용/yolo_model/date_detection.pt")
model_token = YOLO("/Users/hyeongmin_k/세븐일레븐_데이터_코드_제출용/yolo_model/dmy_detection.pt")

BASE_CONF = 0.25
BASE_IOU = 0.2
TOKEN_ENLARGE_SCALE = 1.1
SAVE_ROOT = "/Users/hyeongmin_k/Downloads/12_21"
os.makedirs(SAVE_ROOT, exist_ok=True)


def run_pipeline_on_image(img_path: str, save_root: str = SAVE_ROOT):
    frame = cv2.imread(img_path)
    if frame is None:
        print(f"[❌] 이미지 로드 실패: {img_path}")
        return None

    h, w = frame.shape[:2]
    fname = os.path.splitext(os.path.basename(img_path))[0]
    img_save_dir = os.path.join(save_root, fname)
    os.makedirs(img_save_dir, exist_ok=True)

    # 1) FULL DETECT
    results_full = model_full.predict(
        source=frame,
        conf=BASE_CONF,
        iou=BASE_IOU,
        verbose=False,
        max_det=10,
    )
    full_vis_frame = frame.copy()
    boxes_full = results_full[0].boxes.xyxy.cpu().numpy()
    scores_full = results_full[0].boxes.conf.cpu().numpy()
    classes_full = results_full[0].boxes.cls.cpu().numpy()

    full_candidates = []
    for i, box in enumerate(boxes_full):
        cls_name = full_model_class_to_name(classes_full[i])
        if cls_name == "full":
            full_candidates.append((box, scores_full[i]))

    if len(full_candidates) == 0:
        print("[FAIL] full not detected → stop pipeline")
        cv2.imwrite(os.path.join(img_save_dir, "00_orig.jpg"), frame)
        return None

    full_candidates.sort(key=lambda x: x[1], reverse=True)
    full_box = full_candidates[0][0].astype(int)
    fx1, fy1, fx2, fy2 = full_box
    crop_full = frame[fy1:fy2, fx1:fx2].copy()
    cv2.rectangle(full_vis_frame, (fx1, fy1), (fx2, fy2), (0, 200, 255), 2)
    cv2.putText(full_vis_frame, "FULL", (fx1, max(fy1 - 5, 0)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)

    cv2.imwrite(os.path.join(img_save_dir, "00_orig.jpg"), frame)
    cv2.imwrite(os.path.join(img_save_dir, "01_full_vis.jpg"), full_vis_frame)
    cv2.imwrite(os.path.join(img_save_dir, "02_full_crop.jpg"), crop_full)

    # 2) DATE DETECT
    results_after = model_after.predict(
        source=crop_full,
        conf=BASE_CONF,
        iou=BASE_IOU,
        verbose=False,
        max_det=20,
    )
    boxes_after = results_after[0].boxes.xyxy.cpu().numpy()
    scores_after = results_after[0].boxes.conf.cpu().numpy()
    classes_after = results_after[0].boxes.cls.cpu().numpy()

    date_boxes = []
    for j, abox in enumerate(boxes_after):
        cls_after_name = after_class_to_name(classes_after[j])
        if cls_after_name == "date":
            date_boxes.append((abox, scores_after[j]))

    if len(date_boxes) == 0:
        print("[FAIL] date not detected (after full) → stop pipeline")
        cv2.imwrite(os.path.join(img_save_dir, "03_date_vis_empty.jpg"), crop_full)
        return None

    date_boxes.sort(key=lambda x: x[1], reverse=True)

    after_vis = crop_full.copy()
    for idx, (dbox, dscore) in enumerate(date_boxes):
        ax1, ay1, ax2, ay2 = map(int, dbox)
        color_date = after_class_to_color(0)
        cv2.rectangle(after_vis, (ax1, ay1), (ax2, ay2), color_date, 2)
        cv2.putText(after_vis, f"date#{idx} {dscore:.2f}",
                    (ax1, max(ay1 - 5, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_date, 2)

    cv2.imwrite(os.path.join(img_save_dir, "03_date_vis_all.jpg"), after_vis)

    all_date_results = []

    # 3) 각 date 박스 안에서 토큰 3클래스 검출
    for date_idx, (dbox, dscore) in enumerate(date_boxes):
        ax1, ay1, ax2, ay2 = map(int, dbox)
        crop_date = crop_full[ay1:ay2, ax1:ax2].copy()

        date_save_dir = os.path.join(img_save_dir, f"date_{date_idx:02d}")
        os.makedirs(date_save_dir, exist_ok=True)

        cv2.imwrite(os.path.join(date_save_dir, "00_date_crop.jpg"), crop_date)

        results_token = model_token.predict(
            source=crop_date,
            conf=BASE_CONF,
            iou=BASE_IOU,
            verbose=False,
            max_det=3,
        )
        token_boxes = results_token[0].boxes.xyxy.cpu().numpy()
        token_scores = results_token[0].boxes.conf.cpu().numpy()
        token_classes = results_token[0].boxes.cls.cpu().numpy()

        token_vis = crop_date.copy()
        h_date, w_date = crop_date.shape[:2]

        token_items = []

        if len(token_boxes) > 0:
            for b_idx, b in enumerate(token_boxes):
                kx1, ky1, kx2, ky2 = map(int, b)
                conf_val = float(token_scores[b_idx])
                cls_id = int(token_classes[b_idx])
                tok_name = token_class_to_name(cls_id)
                tok_color = token_class_to_color(cls_id)

                cv2.rectangle(token_vis, (kx1, ky1), (kx2, ky2), tok_color, 2)
                cv2.putText(token_vis, f"{tok_name} {conf_val:.2f}",
                            (kx1, max(ky1 - 5, 0)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, tok_color, 2)

                # 확대해서 OCR
                ex1, ey1, ex2, ey2 = enlarge_box(
                    kx1, ky1, kx2, ky2,
                    scale=TOKEN_ENLARGE_SCALE,
                    img_w=w_date,
                    img_h=h_date
                )
                digit_crop = crop_date[ey1:ey2, ex1:ex2]
                cv2.imwrite(
                    os.path.join(date_save_dir, f"01_tok_{b_idx:02d}_{tok_name}_crop.jpg"),
                    digit_crop
                )

                # OCR
                text_val = infer_ocr_from_np(digit_crop)
                text_clean = clean_token_text(text_val)
                print(f"[OCR RAW] date#{date_idx} {tok_name} -> '{text_val}' (conf={conf_val:.2f})")

                # ✅ Y4인데 2자리만 나온 경우 → NUM2로 다운
                if tok_name == "Y4":
                    if text_clean.isdigit() and len(text_clean) <= 2:
                        tok_name = "NUM2"
                        tok_color = token_class_to_color(2)

                cx = (kx1 + kx2) / 2.0
                cy = (ky1 + ky2) / 2.0
                token_items.append({
                    "cls": tok_name,
                    "text": text_clean,
                    "cx": cx,
                    "cy": cy,
                })
        else:
            print(f"[INFO] date#{date_idx}: token 0개")

        cv2.imwrite(os.path.join(date_save_dir, "02_token_vis.jpg"), token_vis)

        all_date_results.append({
            "date_idx": date_idx,
            "bbox": [ax1, ay1, ax2, ay2],
            "tokens": token_items,
        })

    final_date, extra_dates = postprocess_all_dates(all_date_results, TODAY)

    print(f"[✅ DONE] {img_path} → raw:{len(all_date_results)}개, final:{final_date}, extras:{extra_dates}")

    # ✅ 이제 (대표값, 후보들)로 리턴
    return final_date, extra_dates


########################################
# 5. 검증 루프
########################################
def load_gt_map(txt_path: str):
    gt = {}
    with open(txt_path, "r", encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split(",", 1)
            if len(parts) != 2:
                continue
            img_name = parts[0].strip()
            date_str = parts[1].strip()
            try:
                gt_date = datetime.strptime(date_str, "%Y-%m-%d").date()
            except ValueError:
                continue
            gt[img_name] = gt_date
    return gt


if __name__ == "__main__":
    IMG_DIR = "/Users/hyeongmin_k/세븐일레븐_데이터_코드_제출용/검증용_데이터셋/프로젝트_검증용_사진"
    GT_TXT = "/Users/hyeongmin_k/세븐일레븐_데이터_코드_제출용/검증용_데이터셋/프로젝트_검증용.txt"

    gt_map = load_gt_map(GT_TXT)

    total = 0
    correct = 0
    wrong_cases = []
    skipped = []  # ✅ 집계 제외 목록

    # ✅ (txt에 존재하는 파일명) 기준으로 돌되, 실제 이미지 파일이 없으면 제외
    for fname, gt_date in gt_map.items():
        img_path = os.path.join(IMG_DIR, fname)

        # ✅ 이미지가 없으면 집계 대상 제외
        if not os.path.isfile(img_path):
            skipped.append({"file": fname, "reason": "image_missing"})
            continue

        # ✅ 여기까지 왔으면 (txt 존재 + 이미지 존재) = 집계 대상
        out = run_pipeline_on_image(img_path)

        if out is None:
            pred_date = None
            extra = []
        else:
            pred_date, extra = out

        total += 1

        if (pred_date is not None and pred_date == gt_date) or (gt_date in extra):
            correct += 1
        else:
            wrong_cases.append({
                "file": fname,
                "gt": gt_date,
                "pred": pred_date,
                "alts": extra,
            })

    acc = (correct / total * 100.0) if total > 0 else 0.0
    print(f"\n[RESULT] (txt+image 둘 다 있는 것만) 총 {total}개 중 {correct}개 일치 → 정확도 {acc:.2f}%")

    if skipped:
        print(f"\n[SKIPPED] 집계 제외 {len(skipped)}개")
        # 필요하면 이유별 카운트도 추가 가능
        for s in skipped[:20]:  # 너무 길면 20개만
            print(f" - {s['file']} ({s['reason']})")
        if len(skipped) > 20:
            print(" - ...")

    if wrong_cases:
        print("\n[WRONG LIST]")
        for i, case in enumerate(wrong_cases, 1):
            print("====================================")
            print(f"{i}. 파일: {case['file']}")
            print(f"   GT    : {case['gt']}")
            print(f"   PRED  : {case['pred']}")
            print(f"   ALTS  : {case['alts']}")
        print("====================================")
        print(f"총 {len(wrong_cases)}개 오답/실패")
    else:
        print("\n[WRONG LIST] 🎉 전부 맞았습니다.")



[OCR RAW] date#0 NUM2 -> '08' (conf=0.96)
[OCR RAW] date#0 Y4 -> '2026' (conf=0.94)
[OCR RAW] date#0 NUM2 -> '11' (conf=0.92)
[FINAL] 2026-08-11
[✅ DONE] /Users/hyeongmin_k/세븐일레븐_데이터_코드_제출용/검증용_데이터셋/프로젝트_검증용_사진/KakaoTalk_Photo_2025-11-01-19-52-35 001.jpeg → raw:1개, final:2026-08-11, extras:[]
[OCR RAW] date#0 Y4 -> '2026' (conf=0.96)
[OCR RAW] date#0 NUM2 -> '05' (conf=0.95)
[OCR RAW] date#0 NUM2 -> '22' (conf=0.94)
[FINAL] 2026-05-22
[✅ DONE] /Users/hyeongmin_k/세븐일레븐_데이터_코드_제출용/검증용_데이터셋/프로젝트_검증용_사진/KakaoTalk_Photo_2025-11-01-19-52-36 002.jpeg → raw:1개, final:2026-05-22, extras:[]
[OCR RAW] date#0 NUM2 -> '03' (conf=0.96)
[OCR RAW] date#0 NUM2 -> '08' (conf=0.96)
[OCR RAW] date#0 Y4 -> '2026' (conf=0.96)
[FINAL] 2026-03-08
[✅ DONE] /Users/hyeongmin_k/세븐일레븐_데이터_코드_제출용/검증용_데이터셋/프로젝트_검증용_사진/KakaoTalk_Photo_2025-11-01-19-52-36 003.jpeg → raw:1개, final:2026-03-08, extras:[]
[OCR RAW] date#0 NUM2 -> '09' (